# Current pipeline walkthrough (offline by default)

Run with a Python **3.10+ kernel** in which `intention-grounding` is installed. From the repository root, the existing environment runs the pipeline with `./.venv/bin/python`; Jupyter is optional and must be available in the chosen kernel environment. Launch from the repository root or `notebooks/`. Execute cells in order.

The interpreter and grounder replay responses are **manually authored software-contract fixtures**, not model evaluation. The bundled four-row CSV is **synthetic**. No WikiDBs-10k table files were found in this checkout; the WikiDBs metadata is a future-source record, not local data. The current pipeline has no verified gold intended-action or table-suitability labels for WikiDBs.


In [1]:
import json
import sys
from importlib.resources import files
from pathlib import Path

from intention_grounding.cli import _offline_inputs
from intention_grounding.interpreter.core import interpret
from intention_grounding.action_base import load_action_base, ancestry
from intention_grounding.grounder import ground
from intention_grounding.profiler import profile_table
from intention_grounding.aligner import compare_card

root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "pyproject.toml").is_file()), None)
if root is None:
    raise RuntimeError("Launch from the repository root or notebooks/")
if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ kernel")
print("Repository:", root)
print("Python:", sys.version.split()[0])


Repository: /Users/enaso/Desktop/play ground/Intention
Python: 3.10.19


## 1. Input request

The CLI's offline helper selects the `observations` case from the installed replay fixture. Inspect the actual input before calling the provider.


In [2]:
input_id, request, interpreter_provider, grounder_provider = _offline_inputs()
print({"input_id": input_id, "request": request})


{'input_id': 'observations', 'request': 'Compare France annual income in 2019 versus annual income in 2021.'}


## 2. Interpreter: raw response and validated interpretation

The provider is a local replay object. Its response is never fetched from a network. The raw JSON is distinct from the interpreter's validated envelope.


In [3]:
raw_interpreter = interpreter_provider.invoke(input_id, request)
raw_doc = json.loads(raw_interpreter)
print({"raw_response_bytes": len(raw_interpreter.encode("utf-8")),
       "raw_status": raw_doc["status"],
       "raw_actions": [a["id"] for a in raw_doc["actions"]],
       "raw_evidence": [(e["id"], e["text"]) for e in raw_doc["evidence"]]})


{'raw_response_bytes': 2007, 'raw_status': 'unambiguous', 'raw_actions': ['a1'], 'raw_evidence': [('e1', 'income'), ('e2', 'France'), ('e3', '2019'), ('e4', '2021'), ('e5', 'annual'), ('e6', 'Compare'), ('e7', 'Compare')]}


In [4]:
interpreted = interpret(input_id, request, interpreter_provider)
assert interpreted["failure"] is None
print({"provider": interpreted["metadata"]["provider"],
       "failure": interpreted["failure"],
       "validated_status": interpreted["interpretation"]["status"],
       "validated_roles": [a["role"] for a in interpreted["interpretation"]["actions"][0]["arguments"]]})


{'provider': 'manual-replay', 'failure': None, 'validated_status': 'unambiguous', 'validated_roles': ['measure', 'subject', 'time_points', 'time_points', 'time_grain']}


## 3. Canonical Action Base

The loader checks references and parent links. The draft AB supports the action hierarchy below; the `kb/` knowledge base is from an older workflow.


In [5]:
ab = load_action_base()
print("AB version:", ab["ab_version"])
for action_id, action in ab["actions"].items():
    print(action_id, "parent:", action["parent"], "abstract:", action.get("abstract"))
print("Selected action ancestry:", ancestry("compare.time", ab))


AB version: 0.1.0-draft
compare parent: None abstract: True
compare.groups parent: compare abstract: False
compare.groups.trajectories parent: compare.groups abstract: False
compare.time parent: compare abstract: False
Selected action ancestry: ['compare', 'compare.time']


## 4. Grounder: proposal, validation, card

The proposed action and argument references come from a local replay response. `ground` independently validates the action ID, AB version, inherited roles, cardinality, and evidence references before constructing a card. A grounded card says the request was represented in the AB; it does not say any table is suitable.


In [6]:
raw_grounder = grounder_provider.invoke(interpreted["interpretation"], ab)
print("Raw proposal:", json.loads(raw_grounder))


Raw proposal: {'ab_version': '0.1.0-draft', 'input_id': 'observations', 'action_id': 'compare.time', 'bindings': {'measure': [{'action_id': 'a1', 'argument_index': 0}], 'subject': [{'action_id': 'a1', 'argument_index': 1}], 'time_points': [{'action_id': 'a1', 'argument_index': 2}, {'action_id': 'a1', 'argument_index': 3}], 'time_grain': [{'action_id': 'a1', 'argument_index': 4}]}, 'evidence': ['e1', 'e2', 'e3', 'e4', 'e5']}


In [7]:
grounded = ground(interpreted, ab, grounder_provider, run_id="notebook-offline")
assert grounded["status"] == "grounded", grounded["reasons"]
card = grounded["grounded_card"]
print({"status": grounded["status"],
       "validation": card["deterministic_validation"],
       "selected_action": card["selected_child_action_id"],
       "policy": card["bindings"]["coverage_policy"],
       "requirements": [(r["id"], r["axis"], r["predicate"]) for r in card["requirements"]]})


{'status': 'grounded', 'validation': {'valid': True, 'checks': ['known_action', 'inherited_roles', 'cardinality', 'evidence_refs']}, 'selected_action': 'compare.time', 'policy': ['available_observations'], 'requirements': [('measure_semantics', 'sem', 'measure_semantics'), ('observation_binding', 'str', 'observation_binding'), ('usable_numeric_observations', 'qual', 'usable_numeric_observations'), ('comparable_measure_definition', 'aff', 'comparable_measure_definition'), ('time_coverage', 'cov', 'time_coverage')]}


## 5. Local table profile and provenance

The profiler scans the entire bundled CSV, records observed statistics, and attaches file SHA-256 evidence. The source metadata labels this table as a synthetic fixture.


In [8]:
table_path = Path(files("intention_grounding").joinpath("fixtures/offline_table.csv"))
metadata = json.loads(files("intention_grounding").joinpath("fixtures/offline-source-metadata.json").read_text())
profile = profile_table(table_path, dataset_id=metadata["dataset_id"], source_metadata=metadata)
print({"dataset": metadata["dataset_name"], "table_id": profile["table_id"],
       "rows": profile["row_count"], "sha256": profile["file_sha256"],
       "columns": [{k: c.get(k) for k in ("name", "observed_type", "null_count", "numeric_statistics", "distinct_count")}
                   for c in profile["columns"]],
       "evidence": profile["evidence"][0],
       "semantic_annotations": profile["semantic_annotations"]})


{'dataset': 'offline synthetic contract fixture', 'table_id': 'offline-fixture-v1:e68cc872205230ca', 'rows': 4, 'sha256': 'e68cc872205230cac0a1f5d5b975d8e6680cd6c56e54f1ca3a886cc24095eb87', 'columns': [{'name': 'country', 'observed_type': 'string', 'null_count': 0, 'numeric_statistics': None, 'distinct_count': 2}, {'name': 'year', 'observed_type': 'number', 'null_count': 0, 'numeric_statistics': {'finite_count': 4, 'minimum': 2019.0, 'maximum': 2021.0}, 'distinct_count': None}, {'name': 'income', 'observed_type': 'number', 'null_count': 0, 'numeric_statistics': {'finite_count': 4, 'minimum': 42000.0, 'maximum': 48000.0}, 'distinct_count': None}], 'evidence': {'id': 'column-0', 'fact': 'observed_column_profile', 'column': 'country', 'method': 'full_local_scan_v1', 'scope': 'entire_local_file', 'complete': True, 'source': {'path': '/Users/enaso/Desktop/play ground/Intention/intention_grounding/fixtures/offline_table.csv', 'sha256': 'e68cc872205230cac0a1f5d5b975d8e6680cd6c56e54f1ca3a886cc

## 6. Axis comparison

The current profiler does not establish semantic meanings, units, populations, or observation relations required by the AB predicates. The aligner returns `unknown` for these unsupported checks. This is **not** a successful alignment or suitability finding.


In [9]:
comparison = compare_card(card, [profile])
print({"policy": comparison["policy"],
       "table_state": comparison["tables"][0]["overall_state"],
       "requirements": [(r["axis"], r["predicate"], r["state"], r["reason"])
                        for r in comparison["tables"][0]["requirements"]]})


{'policy': 'all_of_three_valued_v1', 'table_state': 'unknown', 'requirements': [('sem', 'measure_semantics', 'unknown', 'PROFILE_DOES_NOT_MEASURE_PREDICATE'), ('str', 'observation_binding', 'unknown', 'PROFILE_DOES_NOT_MEASURE_PREDICATE'), ('qual', 'usable_numeric_observations', 'unknown', 'PROFILE_DOES_NOT_MEASURE_PREDICATE'), ('aff', 'comparable_measure_definition', 'unknown', 'PROFILE_DOES_NOT_MEASURE_PREDICATE'), ('cov', 'time_coverage', 'unknown', 'PROFILE_DOES_NOT_MEASURE_PREDICATE')]}


## 7. CLI artifact layout

Exact offline command from the repository root:

```bash
./.venv/bin/python -m intention_grounding.cli --mode offline --output outputs/pipeline
```

The next cell runs the same CLI in a temporary output directory and lists the artifact names. The CLI normally creates a unique `run-*` directory under the requested `--output` path. It makes no network calls in offline mode.


In [10]:
import subprocess
import tempfile

with tempfile.TemporaryDirectory(prefix="intention-walkthrough-") as temp_dir:
    result = subprocess.run(
        [sys.executable, "-m", "intention_grounding.cli", "--mode", "offline",
         "--output", temp_dir],
        cwd=root, text=True, capture_output=True, check=True)
    runs = list(Path(temp_dir).glob("run-*"))
    assert len(runs) == 1
    manifest = json.loads((runs[0] / "run.json").read_text())
    print(result.stdout.strip())
    print("Artifact files:", sorted(p.name for p in runs[0].iterdir()))
    print("Manifest mode / AB version:", manifest["mode"], manifest["ab_version"])


run_id=run-bc2658a94275 interpretation=unambiguous grounding=grounded tables=1 alignment=unknown
artifacts=/var/folders/96/ftrw9x3x6pb1vd2dfq__n0k80000gp/T/intention-walkthrough-j89t7bdk/run-bc2658a94275
Artifact files: ['comparison.json', 'grounder.raw.txt', 'grounding.json', 'interpretation.json', 'interpreter.raw.txt', 'run.json', 'table-profiles.json']
Manifest mode / AB version: offline 0.1.0-draft


## 8. Optional live Groq call — explicit opt-in

This cell is skipped unless you edit `RUN_LIVE_GROQ` to `True`, install the optional SDK, and configure `GROQ_API_KEY` and `GROQ_MODEL_ID` in `intention_grounding/.env` or the process environment. It never prints the key. A live invocation makes external API calls and may incur charges. Leave the default flag unchanged for a fully offline walkthrough.


In [11]:
RUN_LIVE_GROQ = False

if RUN_LIVE_GROQ:
    from intention_grounding.groq_config import groq_api_key, groq_model_id
    from intention_grounding.interpreter.providers import GroqProvider
    from intention_grounding.grounder import GroqGrounderProvider

    model = groq_model_id()
    groq_api_key()
    live_interpretation = interpret(input_id, request, GroqProvider(model=model))
    live_grounding = ground(live_interpretation, ab, GroqGrounderProvider(model=model),
                            run_id="notebook-live")
    print({"interpretation_failure": live_interpretation["failure"],
           "interpretation_status": (live_interpretation["interpretation"] or {}).get("status"),
           "grounding_status": live_grounding["status"],
           "grounding_reasons": live_grounding["reasons"]})
else:
    print("Skipped live Groq calls.")


Skipped live Groq calls.
